In [16]:
import pandas as pd

# Parameters
year = 2024
columns_to_keep = ["locatie_code","grootheid_code", "parameter_code", "hoedanigheid_code", "eventdatum", "event_aquocode", "eenheid_code"]

file_path = rf"C:\Ocean\Work\Projects\2026\ISC\Tools\Repositories\voorbeeld\isc_2023-2025\ISC_{year}.xlsx"
sheet_name = str(year)

# Read the sheet into a DataFrame
df = pd.read_excel(file_path, sheet_name=sheet_name)
df = df[columns_to_keep].copy()

# Enforce one row per unique combination
rows_before = len(df)
df = df.drop_duplicates(subset=columns_to_keep, keep="first").reset_index(drop=True)
removed_rows = rows_before - len(df)

# Safety check
if df.duplicated(subset=columns_to_keep).any():
    raise ValueError("Duplicate combinations are still present after deduplication.")

print(f"Removed {removed_rows} duplicate rows.")
df

Removed 0 duplicate rows.


,locatie_code,grootheid_code,parameter_code,hoedanigheid_code,eventdatum,event_aquocode,eenheid_code
0,hansweert.geul,ANMLE,Gd,nf,2024-01-09,0,DIMSLS
1,hansweert.geul,ANMLE,Gd,nf,2024-01-29,0,DIMSLS
2,hansweert.geul,ANMLE,Gd,nf,2024-02-26,0,DIMSLS
3,hansweert.geul,ANMLE,Gd,nf,2024-03-25,0,DIMSLS
4,hansweert.geul,ANMLE,Gd,nf,2024-04-22,0,DIMSLS
...,...,...,...,...,...,...,...
33398,wissenkerke,pH,NVT,NVT,2024-09-03,0,DIMSLS
33399,wissenkerke,pH,NVT,NVT,2024-09-18,0,DIMSLS
33400,wissenkerke,pH,NVT,NVT,2024-10-09,0,DIMSLS
33401,wissenkerke,pH,NVT,NVT,2024-11-12,0,DIMSLS


In [ ]:
import pandas as pd

year = 2024
columns_to_keep = ["locatie_code","grootheid_code", "parameter_code", "hoedanigheid_code", "eventdatum", "event_aquocode", "eenheid_code"]

file_path = rf"C:\Ocean\Work\Projects\2026\ISC\Tools\Repositories\voorbeeld\isc_2023-2025\ISC_{year}.xlsx"
all_locs = pd.read_excel(r"C:\Ocean\Work\Projects\2026\ISC\Tools\Repositories\mappings\locations-mapped.xlsx")
sheet_name = str(year)

df = pd.read_excel(file_path, sheet_name=sheet_name)
df = df[columns_to_keep].copy()

# Mapping column names
loc_col = "locatie_code"
station_id_col = "Identitication unique de la station"  

# Merge station unique ID into df
mapping = all_locs[[loc_col, station_id_col]].drop_duplicates(subset=[loc_col])
df = df.merge(mapping, on=loc_col, how="left")

df

Unique locatie_code values:
hansweert.geul
sasvangent.grens
schaarvanoudendoel
terneuzen.boei20
vlissingen.boeissvh
walcheren.2kmuitdekust
wissenkerke

Aantal unieke locatie_code: 7


,locatie_code,grootheid_code,parameter_code,hoedanigheid_code,eventdatum,event_aquocode,eenheid_code,Identitication unique de la station
0,hansweert.geul,ANMLE,Gd,nf,2024-01-09,0,DIMSLS,NL_HANSWGL
1,hansweert.geul,ANMLE,Gd,nf,2024-01-29,0,DIMSLS,NL_HANSWGL
2,hansweert.geul,ANMLE,Gd,nf,2024-02-26,0,DIMSLS,NL_HANSWGL
3,hansweert.geul,ANMLE,Gd,nf,2024-03-25,0,DIMSLS,NL_HANSWGL
4,hansweert.geul,ANMLE,Gd,nf,2024-04-22,0,DIMSLS,NL_HANSWGL
...,...,...,...,...,...,...,...,...
33398,wissenkerke,pH,NVT,NVT,2024-09-03,0,DIMSLS,NL89_WISSKKE
33399,wissenkerke,pH,NVT,NVT,2024-09-18,0,DIMSLS,NL89_WISSKKE
33400,wissenkerke,pH,NVT,NVT,2024-10-09,0,DIMSLS,NL89_WISSKKE
33401,wissenkerke,pH,NVT,NVT,2024-11-12,0,DIMSLS,NL89_WISSKKE


In [24]:
import pandas as pd

# If ds is not defined yet:
ds = df.copy()

mapping_path = r"C:\Ocean\Work\Projects\2026\ISC\Tools\Repositories\mappings\parameter_mapping_final.xlsx"
mapping_sheet = "wadar_isc_cas"

mapping_df = pd.read_excel(mapping_path, sheet_name=mapping_sheet)

# ds column and equivalent mapping key
ds_parameter_col = "parameter_code"
mapping_parameter_col = "wadar_PARCode"

# Column you want to bring from mapping table
mapping_value_col = "Unieke identificatie gemeten parameter"  # adjust exact spelling if needed
new_col_in_ds = "mapped_parameter_id"

# Validate columns
for col in [ds_parameter_col]:
    if col not in ds.columns:
        raise KeyError(f"{col} not found in ds. Available: {list(ds.columns)}")

for col in [mapping_parameter_col, mapping_value_col]:
    if col not in mapping_df.columns:
        raise KeyError(f"{col} not found in mapping table. Available: {list(mapping_df.columns)}")

# Keep one row per wadar_PARCode
mapping_clean = (
    mapping_df[[mapping_parameter_col, mapping_value_col]]
    .dropna(subset=[mapping_parameter_col])
    .drop_duplicates(subset=[mapping_parameter_col], keep="first")
)

# Map value into ds
map_dict = mapping_clean.set_index(mapping_parameter_col)[mapping_value_col]
ds[new_col_in_ds] = ds[ds_parameter_col].map(map_dict)

# Stats
n_mapping_values = mapping_clean[mapping_value_col].dropna().nunique()
n_unique_ds_values = ds[new_col_in_ds].dropna().nunique()
n_empty_rows = ds[new_col_in_ds].isna().sum()

print(f"Unique values in mapping table ({mapping_value_col}): {n_mapping_values}")
print(f"Unique mapped values in ds ({new_col_in_ds}): {n_unique_ds_values}")
print(f"Rows not mapped (empty {new_col_in_ds}): {n_empty_rows}")

# Optional: show unmapped parameter_code values
unmapped_codes = sorted(ds.loc[ds[new_col_in_ds].isna(), ds_parameter_col].dropna().unique())
print(f"Unmapped unique {ds_parameter_col} values: {len(unmapped_codes)}")
print(unmapped_codes[:20])

Unique values in mapping table (Unieke identificatie gemeten parameter): 78
Unique mapped values in ds (mapped_parameter_id): 78
Rows not mapped (empty mapped_parameter_id): 25917
Unmapped unique parameter_code values: 380
['111TClC2a', '1122T4ClC2a', '112TClC2a', '11ClPF3OUdS', '11DClC2a', '11DClC2e', '123TC1yBen', '123TClC3a', '123benztazl', '124TC1yBen', '12DClBen', '12DClC3a', '12xyln', '135TC1yBen', '135TNO2Ben', '13DClBen', '13DClC3a', '13DNO2Ben', '14DClBen', '17bestDol']


In [ ]:


# How many rows share each exact combination
group_sizes = (
    df.groupby(columns_to_keep, dropna=False)
    .size()
    .rename("duplicate_count")
)

df = df.merge(group_sizes, on=columns_to_keep, how="left")

# List of row indices with the same combination (including this row)
def indices_in_group(g):
    return list(g.index)

matching = (
    df.groupby(columns_to_keep, dropna=False)
    .apply(indices_in_group, include_groups=False)
    .rename("matching_row_indices")
    .reset_index()
)

df = df.merge(matching, on=columns_to_keep, how="left")

# Optional: only "other" rows (exclude current index)
df["other_matching_row_indices"] = df.apply(
    lambda r: [i for i in r["matching_row_indices"] if i != r.name],
    axis=1,
)

# Flag: unique vs duplicate
df["is_duplicate"] = df["duplicate_count"] > 1

# Create df with is_duplicate = True
duplicates_df = df[df["is_duplicate"]].copy()
duplicates_df

# Save all duplicate rows from the original sheet, grouped together by duplicate keys
source_df = pd.read_excel(file_path, sheet_name=sheet_name)
dup_mask = source_df.duplicated(subset=columns_to_keep, keep=False)

# Duplicate metadata per key combination
duplicate_counts = (
    source_df.groupby(columns_to_keep, dropna=False)
    .size()
    .rename("duplicate_count")
    .reset_index()
)

matching_rows = (
    source_df.reset_index()
    .groupby(columns_to_keep, dropna=False)["index"]
    .apply(list)
    .rename("matching_row_indices")
    .reset_index()
)

all_duplicates_sorted = (
    source_df.loc[dup_mask]
    .reset_index()
    .rename(columns={"index": "original_index"})
    .merge(duplicate_counts, on=columns_to_keep, how="left")
    .merge(matching_rows, on=columns_to_keep, how="left")
    .sort_values(by=columns_to_keep + ["original_index"], kind="stable")
    .reset_index(drop=True)
)

output_csv = rf"C:\Ocean\Work\Projects\2026\ISC\Tools\Repositories\voorbeeld\isc_2023-2025\all_duplicates_{year}_grootheid_event_aquocode_eenheid_code.csv"
all_duplicates_sorted.to_csv(output_csv, index=False, sep=';')
all_duplicates_sorted

,original_index,locatie_code,locatienaam,locaties_locatie_type,locatie_lat_etrs89,locatie_lon_etrs89,grootheid_code,parameter_code,hoedanigheid_code,compartiment_code,...,event_resultaattijd,event_waarde,event_waarde_limietsymbool,event_waarde_tekst,status_waarde,event_aquocode,inventarisaties_code,inventarisaties_omschrijving,duplicate_count,matching_row_indices


In [1]:



donar_isc_cas = pd.read_csv(Path.joinpath(p.parent,'mappings/donar-isc-cas.csv'))
all_locs = pd.read_excel(Path.joinpath(p.parent,'mappings/locations-mapped.xlsx'))
# %%
donar = donar[donar['LOCOMS'].isin(all_locs['LOCOMS'].to_list())] #filter op locaties
donar = donar[donar['KWC'] <= 50] # filter op kwaliteit
# %%
# daggemiddelden berekenen door de location + parameter combinatie 
# mappen naar de parameter mapping
donarmap = donar.merge(donar_isc_cas, left_on='PAROMS', right_on = 'DONAR_Parameter', how='inner')

# Ensure strings and pad TIME to 4 digits (HHMM)
date_str = donarmap["DATUM"].astype(str).str.zfill(8)      # '20240109'
time_str = donarmap["TIJD"].astype(str).str.zfill(4)      # '1250', '0905', '0005'

# Concatenate and parse
donarmap["timestamp"] = pd.to_datetime(date_str + time_str, format="%Y%m%d'%H%M", errors="coerce")
donarmap = donarmap.drop(columns=["DATUM", "TIJD"])

# 1) Remove all characters except digits, +, -, ., e/E
clean = donarmap["WAARDE"].astype(str).str.replace(r"[^0-9eE+\-\.]", "", regex=True)

# 2) Convert to numeric (scientific notation is supported)
donarmap["value"] = pd.to_numeric(clean, errors="coerce")
# %%
donarmap["date"] = donarmap["timestamp"].dt.floor("D")

daily = (
    donarmap
    .groupby(["LOCOMS", "PAROMS", "date"], as_index=False)["value"]
    .mean()
    .rename(columns={"value": "value_daily_mean"})
)


FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Ocean\\Work\\Projects\\2026\\ISC\\voorbeeld\\isc2024\\isc2024.csv'